# Using o-isochrone as a Python library

This notebook shows the **core library** — the same code the example API and MapLibre demo call.

1. **Single origin** — `compute_isochrone` returns one GeoJSON FeatureCollection.
2. **Batch of 3 origins** — `compute_isochrones` / `compute_isochrones_async` return a list of FeatureCollections.
   Each collection (and every feature) is stamped with `origin_id` so overlapping isochrones can be filtered.

The first run downloads or clips OSM roads around each point (Overpass or a local PBF from `.env`). Later runs reuse the OSMnx cache.

```bash
uv sync --extra api --extra notebooks
uv run jupyter notebook notebook_examples/isochrone_usage.ipynb
```

In [ ]:
from oisochrone import (
    IsochroneOrigin,
    compute_isochrone,
    compute_isochrones,
    compute_isochrones_async,
)

## Single point

Milan Duomo, 12 minutes on foot. `include` can be a comma-separated string or a list of layer names (`buffered_lines`, `lines`, `filled_buffer`, `convex_hull`, `concave_hull`, `h3`).

In [ ]:
single = compute_isochrone(
    lat=45.4642,
    lon=9.1900,
    travel_time_min=12,
    travel_type="foot",
    include="buffered_lines",
    buffer_meters=10,
    origin_id="duomo",  # optional; stamped on metadata + feature properties
)

print(single["type"], "with", len(single["features"]), "feature(s)")
print("metadata:", single["metadata"])
print("feature kinds:", [f["properties"]["kind"] for f in single["features"]])

## Batch: three overlapping origins

`IsochroneOrigin` is the object class for each starting point. Shared travel parameters apply to every origin; an origin may override them. Leave `id` blank to get incremental ids (`"1"`, `"2"`, …), skipping any ids you already set.

These three Milan points sit close enough that their 12-minute walk isochrones overlap — `origin_id` is how you tell the layers apart.

In [ ]:
origins = [
    IsochroneOrigin(lat=45.4642, lon=9.1900, id="duomo"),
    IsochroneOrigin(lat=45.4863, lon=9.2045),          # id becomes "1"
    IsochroneOrigin(lat=45.4515, lon=9.1750, id="navigli"),
]

collections = compute_isochrones(
    origins,
    travel_time_min=12,
    travel_type="foot",
    include=["buffered_lines"],
    max_concurrency=3,  # thread pool; safe to call from a notebook
)

for collection in collections:
    meta = collection["metadata"]
    print(
        f"origin_id={meta['origin_id']!r}  "
        f"({meta['lat']}, {meta['lon']})  "
        f"{len(collection['features'])} feature(s)"
    )

Filter one origin out of the merged layer set — the same idea the demo map uses:

In [ ]:
navigli_only = [
    feature
    for collection in collections
    for feature in collection["features"]
    if feature["properties"].get("origin_id") == "navigli"
]
print("features for navigli:", len(navigli_only))

### Async parallel batch

From async code (including Jupyter, which already runs an event loop) prefer `compute_isochrones_async`. It uses `asyncio.to_thread` with a concurrency cap.

In [ ]:
async_collections = await compute_isochrones_async(
    origins,
    travel_time_min=12,
    travel_type="foot",
    include="buffered_lines",
    max_concurrency=3,
)
assert [c["metadata"]["origin_id"] for c in async_collections] == [
    c["metadata"]["origin_id"] for c in collections
]
print("async origin ids:", [c["metadata"]["origin_id"] for c in async_collections])

## Same library through the example API

The FastAPI service is a thin wrapper: `POST /isochrone` for one origin, `POST /isochrone/batch` for many. Uncomment if the API is running (`uv run uvicorn api.main:app --port 8000` or `docker compose up`).

In [ ]:
# import httpx
#
# API = "http://127.0.0.1:8000"  # docker compose publishes 8001
# with httpx.Client(timeout=300.0) as client:
#     one = client.post(f"{API}/isochrone", json={
#         "lat": 45.4642, "lon": 9.19,
#         "travel_time_min": 12, "travel_type": "foot",
#         "include": ["buffered_lines"], "origin_id": "duomo",
#     }).json()
#     batch = client.post(f"{API}/isochrone/batch", json={
#         "origins": [
#             {"lat": 45.4642, "lon": 9.19, "id": "duomo"},
#             {"lat": 45.4863, "lon": 9.2045},
#             {"lat": 45.4515, "lon": 9.175, "id": "navigli"},
#         ],
#         "travel_time_min": 12,
#         "travel_type": "foot",
#         "include": ["buffered_lines"],
#     }).json()
# print(one["metadata"]["origin_id"], batch["count"])